# CyberTrader — Backtest Analysis

This notebook provides interactive backtest analysis including:
- Strategy comparison across multiple time ranges
- Factor score visualization
- Performance metrics (Sharpe, drawdown, win rate)
- Equity curve plotting

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime

from cyber_trader.data.catalog import get_catalog
from cyber_trader.engines.backtest import BacktestConfig, BacktestRunner
from cyber_trader.config import get_settings

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('Setup complete')

## 1. Inspect Available Data

In [ ]:
from nautilus_trader.model.data import Bar

catalog = get_catalog()
settings = get_settings()
print(f'Catalog path: {settings.data_catalog_path}')

# List available bar types
try:
    bar_types = catalog.list_data_types()
    print('Available data:', bar_types)
except Exception as e:
    print(f'Catalog may be empty: {e}')

## 2. Load and Plot OHLCV Data

In [ ]:
from nautilus_trader.model.data import Bar, BarType

INSTRUMENT_ID = 'ETH-USDT-SWAP.OKX'
BAR_TYPE = 'ETH-USDT-SWAP.OKX-1-HOUR-LAST-EXTERNAL'

try:
    bars = catalog.bars([BAR_TYPE])
    df = pd.DataFrame([
        {
            'ts': pd.Timestamp(b.ts_event, unit='ns', tz='UTC'),
            'open': b.open.as_double(),
            'high': b.high.as_double(),
            'low': b.low.as_double(),
            'close': b.close.as_double(),
            'volume': b.volume.as_double(),
        }
        for b in bars
    ])
    df.set_index('ts', inplace=True)
    print(f'Loaded {len(df):,} bars from {df.index[0]} to {df.index[-1]}')
    df.tail()
except Exception as e:
    print(f'Could not load bars: {e}')
    print('Run: python scripts/download_data.py --symbol ETH-USDT --timeframe 1h --start 2024-01-01')

In [ ]:
# Price chart
if 'df' in dir() and not df.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(df.index, df['close'], linewidth=1, color='steelblue')
    ax1.set_title(f'{INSTRUMENT_ID} — Close Price', fontsize=14)
    ax1.set_ylabel('Price (USDT)')
    ax2.bar(df.index, df['volume'], width=0.04, color='steelblue', alpha=0.5)
    ax2.set_ylabel('Volume')
    plt.tight_layout()
    plt.show()

## 3. Run Backtests & Compare Strategies

In [ ]:
STRATEGIES = [
    {
        'name': 'TrendFollowing',
        'path': 'cyber_trader.strategies.trend_following:TrendFollowingStrategy',
        'config_path': 'cyber_trader.strategies.trend_following:TrendFollowingConfig',
        'params': {
            'ema_fast': 9, 'ema_slow': 21,
            'macd_fast': 12, 'macd_slow': 26, 'macd_signal': 9,
            'risk_per_trade': 0.01, 'stop_loss_pct': 0.02, 'take_profit_pct': 0.04,
        }
    },
    {
        'name': 'MeanReversion',
        'path': 'cyber_trader.strategies.mean_reversion:MeanReversionStrategy',
        'config_path': 'cyber_trader.strategies.mean_reversion:MeanReversionConfig',
        'params': {
            'bb_period': 20, 'bb_k': 2.0, 'rsi_period': 14,
            'risk_per_trade': 0.01, 'stop_loss_pct': 0.015, 'take_profit_pct': 0.03,
        }
    },
    {
        'name': 'Momentum',
        'path': 'cyber_trader.strategies.momentum:MomentumStrategy',
        'config_path': 'cyber_trader.strategies.momentum:MomentumConfig',
        'params': {
            'mom_period': 10, 'vol_period': 20,
            'risk_per_trade': 0.01, 'stop_loss_pct': 0.02, 'take_profit_pct': 0.05,
        }
    },
]

runner = BacktestRunner()
results = {}

for s in STRATEGIES:
    cfg = BacktestConfig(
        strategy_path=s['path'],
        config_path=s['config_path'],
        strategy_config=s['params'],
        instrument_id=INSTRUMENT_ID,
        bar_type=BAR_TYPE,
        start='2024-01-01',
        end='2024-12-31',
        starting_balance=10_000.0,
    )
    try:
        result = runner.run(cfg)
        results[s['name']] = result.stats()
        print(f"✓ {s['name']} complete")
    except Exception as e:
        print(f"✗ {s['name']} failed: {e}")

comparison = pd.DataFrame(results).T
comparison

In [ ]:
# Strategy comparison chart
if not comparison.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    metrics = ['total_return_pct', 'sharpe_ratio', 'max_drawdown_pct']
    titles = ['Total Return (%)', 'Sharpe Ratio', 'Max Drawdown (%)']
    colors = ['steelblue', 'green', 'red']
    for ax, metric, title, color in zip(axes, metrics, titles, colors):
        if metric in comparison.columns:
            comparison[metric].plot(kind='bar', ax=ax, color=color, alpha=0.7)
            ax.set_title(title)
            ax.set_xlabel('')
            ax.tick_params(axis='x', rotation=15)
    plt.suptitle('Strategy Comparison', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Factor Score Analysis

In [ ]:
# Visualize factor scores on historical data
from cyber_trader.indicators.factor_engine import (
    EMACrossoverFactor, MACDFactor, RSIFactor, BollingerFactor, FactorEngine
)
from nautilus_trader.model.data import Bar, BarType, BarSpecification
from nautilus_trader.model.enums import BarAggregation, PriceType, AggregationSource
from nautilus_trader.model.identifiers import InstrumentId, Symbol, Venue
from nautilus_trader.model.objects import Price, Quantity

if 'df' in dir() and not df.empty:
    # Build factors
    ema_f = EMACrossoverFactor(fast=9, slow=21)
    macd_f = MACDFactor(fast=12, slow=26, signal=9)
    rsi_f = RSIFactor(period=14)
    bb_f = BollingerFactor(period=20, k=2.0)
    
    engine = FactorEngine([ema_f, macd_f, rsi_f, bb_f], long_threshold=0.3, short_threshold=-0.3)
    
    instrument_id = InstrumentId(Symbol('ETH-USDT-SWAP'), Venue('OKX'))
    bar_type = BarType(
        instrument_id=instrument_id,
        spec=BarSpecification(1, BarAggregation.HOUR, PriceType.LAST),
        aggregation_source=AggregationSource.EXTERNAL,
    )
    
    scores_data = []
    for ts, row in df.iterrows():
        bar = Bar(
            bar_type=bar_type,
            open=Price(row['open'], 2),
            high=Price(row['high'], 2),
            low=Price(row['low'], 2),
            close=Price(row['close'], 2),
            volume=Quantity(row['volume'], 2),
            ts_event=int(ts.timestamp() * 1e9),
            ts_init=int(ts.timestamp() * 1e9),
        )
        engine.update(bar)
        if engine.is_initialized:
            scores_data.append({
                'ts': ts,
                'composite': engine.composite_score(),
                **engine.factor_scores(),
            })
    
    scores_df = pd.DataFrame(scores_data).set_index('ts')
    print(f'Generated {len(scores_df):,} score rows')
    scores_df.tail()

In [ ]:
if 'scores_df' in dir() and not scores_df.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Price
    ax1.plot(df.index[-len(scores_df):], df['close'].iloc[-len(scores_df):], color='steelblue', lw=1)
    
    # Mark long/short signals
    longs = scores_df[scores_df['composite'] >= 0.3]
    shorts = scores_df[scores_df['composite'] <= -0.3]
    ax1.scatter(longs.index, df.loc[longs.index, 'close'], marker='^', color='lime', zorder=5, s=30, label='Long')
    ax1.scatter(shorts.index, df.loc[shorts.index, 'close'], marker='v', color='red', zorder=5, s=30, label='Short')
    ax1.legend()
    ax1.set_title('ETH-USDT Price with Factor Signals')
    
    # Composite score
    ax2.plot(scores_df.index, scores_df['composite'], color='purple', lw=1)
    ax2.axhline(0.3, color='lime', linestyle='--', lw=0.8, label='Long threshold')
    ax2.axhline(-0.3, color='red', linestyle='--', lw=0.8, label='Short threshold')
    ax2.axhline(0, color='gray', linestyle='-', lw=0.5)
    ax2.fill_between(scores_df.index, scores_df['composite'], 0,
                     where=scores_df['composite'] > 0, alpha=0.2, color='lime')
    ax2.fill_between(scores_df.index, scores_df['composite'], 0,
                     where=scores_df['composite'] < 0, alpha=0.2, color='red')
    ax2.set_ylim(-1.1, 1.1)
    ax2.set_ylabel('Composite Score')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()